# <font color="#418FDE" size="6.5" uppercase>**Stochastic Gradient**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Train SGD-based classifiers and regressors with appropriate losses and penalties. 
- Configure learning rates, early stopping, and regularization for stable training. 
- Use partial_fit and sparse inputs for incremental or large-scale workflows. 


## **1. SGD Estimators**

### **1.1. SGDClassifier Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_01_01.jpg?v=1783996964" width="250">



>* Efficient linear classifier for large datasets
>* Learns feature weights for class decisions

>* Loss choice defines classifier behavior
>* Penalties control weights and overfitting

>* Scale features for stable SGD updates
>* Efficient for sparse, scalable classification



In [ ]:
#@title Python Code - SGDClassifier Basics

# Train a basic SGDClassifier on scaled numeric features.
# Compare hinge and log loss behavior simply.
# Print accuracy and learned feature weights.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged binary classification dataset.
data = load_breast_cancer()
X = data.data[:, :4]
y = data.target

# Validate the selected feature matrix and target length.
if X.shape[0] != y.shape[0] or X.shape[1] != 4:
    raise ValueError("Expected matching rows and four selected features.")

# Split data before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Hinge loss makes SGDClassifier act like a linear SVM.
hinge_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="hinge", penalty="l2", max_iter=2000, random_state=42),
)

# Log loss makes SGDClassifier learn probability-style scores.
log_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", penalty="l2", max_iter=2000, random_state=42),
)

# Fit each model once on the same training data.
hinge_model.fit(X_train, y_train)
log_model.fit(X_train, y_train)

# Evaluate both models on held-out test data.
hinge_accuracy = accuracy_score(y_test, hinge_model.predict(X_test))
log_accuracy = accuracy_score(y_test, log_model.predict(X_test))

# Inspect the first four learned linear weights.
weights = log_model.named_steps["sgdclassifier"].coef_[0]
rounded_weights = [round(value, 3) for value in weights]

print("scikit-learn version:", sklearn.__version__)
print("Dataset: breast cancer, first 4 numeric features")
print("Hinge loss test accuracy:", round(hinge_accuracy, 3))
print("Log loss test accuracy:", round(log_accuracy, 3))
print("Log loss learned weights:", rounded_weights)



### **1.2. SGD Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_01_02.jpg?v=1783996966" width="250">



>* Linear predictions for continuous targets
>* Small updates enable efficient large-scale training

>* Choose losses for errors and outliers
>* Use penalties to control coefficient complexity

>* Scale features for balanced SGD updates
>* Sparse inputs enable efficient large-scale regression



In [ ]:
#@title Python Code - SGD Regression

# This example trains an SGD regression model.
# Scaling helps stochastic updates behave more steadily.
# The output compares predictions with true values.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small deterministic regression dataset.
features, target = make_regression(
    n_samples=600,
    n_features=6,
    noise=18.0,
    random_state=42,
)

# Check that features and targets match correctly.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Split data before fitting the scaler and model.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Combine scaling with an SGD regressor.
model = make_pipeline(
    StandardScaler(),
    SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=0.0001,
        learning_rate="invscaling",
        eta0=0.01,
        max_iter=2000,
        tol=0.001,
        random_state=42,
    ),
)

# Fit the model using stochastic gradient descent.
model.fit(X_train, y_train)

# Evaluate predictions on unseen test data.
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

# Show concise results for beginners.
print("scikit-learn version:", sklearn.__version__)
print("Test mean absolute error:", round(mae, 2))
print("First 5 true targets:", np.round(y_test[:5], 1).tolist())
print("First 5 predictions:", np.round(predictions[:5], 1).tolist())



### **1.3. Loss Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_01_03.jpg?v=1783996968" width="250">



>* Loss guides SGD weight updates
>* Hinge margins or logistic probabilities

>* Match regression loss to error costs
>* Use robust losses when outliers dominate

>* Loss fits data; penalties control complexity
>* Match both to task and goals



In [ ]:
#@title Python Code - Loss Functions

# Compare SGD losses on one classification dataset.
# Loss choice changes the learned decision boundary.
# The plot shows different linear separators.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small two-feature classification dataset.
features, labels = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

# Check that the example has the expected shape.
if features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 features.")

# Split data so accuracy is measured on unseen examples.
train_x, test_x, train_y, test_y = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Train hinge loss, which behaves like a linear SVM.
hinge_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="hinge", alpha=0.001, max_iter=2000, random_state=42)
)
hinge_model.fit(train_x, train_y)

# Train log loss, which supports probability estimates.
log_model = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", alpha=0.001, max_iter=2000, random_state=42)
)
log_model.fit(train_x, train_y)

# Print concise results that connect loss choice to behavior.
print("scikit-learn version:", sklearn.__version__)
print("Hinge loss test accuracy:", round(hinge_model.score(test_x, test_y), 3))
print("Log loss test accuracy:", round(log_model.score(test_x, test_y), 3))
print("Log loss probability for first test row:", round(log_model.predict_proba(test_x[:1])[0, 1], 3))

# Build a small grid for drawing each decision boundary.
x_min = features[:, 0].min() - 1
x_max = features[:, 0].max() + 1
y_min = features[:, 1].min() - 1
y_max = features[:, 1].max() + 1

# Predict class scores across the grid.
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
grid = np.c_[xx.ravel(), yy.ravel()]
hinge_scores = hinge_model.decision_function(grid).reshape(xx.shape)
log_scores = log_model.decision_function(grid).reshape(xx.shape)

# Plot both learned separators on the same axes.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(features[:, 0], features[:, 1], c=labels, cmap="coolwarm", s=25)
ax.contour(xx, yy, hinge_scores, levels=[0], colors="black", linewidths=2)
ax.contour(xx, yy, log_scores, levels=[0], colors="green", linewidths=2)

# Label the plot for beginner interpretation.
ax.set_title("SGDClassifier loss functions change the boundary")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.plot([], [], color="black", label="hinge boundary")
ax.plot([], [], color="green", label="log_loss boundary")
ax.legend()
plt.show()



## **2. Training Control**

### **2.1. Regularization Penalties**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_02_01.jpg?v=1783996977" width="250">



>* Penalties discourage overly large feature weights
>* Regularization reduces overfitting and stabilizes training

>* Shrinkage keeps many weak features useful
>* Sparsity selects features; combined penalties balance both

>* Tune penalties to balance fit and generalization
>* Stable weights support noisy, ongoing training



In [ ]:
#@title Python Code - Regularization Penalties

# Compare SGD regularization penalties on noisy features.
# Penalties control weight size and feature selection.
# Results show accuracy and learned coefficient sparsity.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create data with many noisy features.
features, target = make_classification(
    n_samples=1200,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    random_state=42,
)

# Check that the generated data has the expected shape.
if features.shape != (1200, 20):
    raise ValueError("Expected 1200 rows and 20 features.")

# Split data before scaling to avoid leakage.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Try three common penalties with the same learning controls.
penalties = ["l2", "l1", "elasticnet"]
print("scikit-learn version:", sklearn.__version__)
print("Penalty      Accuracy  Nonzero weights  Mean |weight|")

for penalty_name in penalties:
    classifier = SGDClassifier(
        loss="log_loss",
        penalty=penalty_name,
        alpha=0.01,
        learning_rate="optimal",
        max_iter=1000,
        tol=1e-3,
        random_state=42,
    )

    model = make_pipeline(StandardScaler(), classifier)
    model.fit(train_features, train_target)

    predictions = model.predict(test_features)
    accuracy = accuracy_score(test_target, predictions)

    weights = model.named_steps["sgdclassifier"].coef_[0]
    nonzero_count = int(np.count_nonzero(np.abs(weights) > 1e-8))
    mean_abs_weight = float(np.mean(np.abs(weights)))

    print(
        f"{penalty_name:<12} {accuracy:.3f}     {nonzero_count:>2}              "
        f"{mean_abs_weight:.3f}"
    )



### **2.2. Learning Rate Schedules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_02_02.jpg?v=1783996979" width="250">



>* Learning rate controls each update step
>* Schedules balance fast progress with stability

>* Schedules control fixed or shrinking update sizes
>* Choose based on data, noise, and regularization

>* Tune schedules with regularization and stopping
>* Monitor losses for stable production updates



In [ ]:
#@title Python Code - Learning Rate Schedules

# Compare learning rate schedules for SGD training.
# Stable schedules reduce noisy validation changes.
# The plot shows accuracy across epochs.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small binary classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    random_state=42,
)

# Split before scaling to avoid data leakage.
X_train, X_valid, y_train, y_valid = train_test_split(
    features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Fit scaling only on the training data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Validate the expected two-dimensional feature table.
if X_train_scaled.shape[1] != 12:
    raise ValueError("Expected 12 scaled feature columns.")

# Define two schedules with the same model family.
schedules = {
    "constant": {"learning_rate": "constant", "eta0": 0.05},
    "adaptive": {"learning_rate": "adaptive", "eta0": 0.05},
}

# Track validation accuracy after each training epoch.
epochs = np.arange(1, 21)
accuracy_history = {}

for schedule_name, settings in schedules.items():
    model = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0005,
        max_iter=1,
        tol=None,
        shuffle=True,
        random_state=42,
        **settings,
    )
    scores = []
    for epoch in epochs:
        model.partial_fit(X_train_scaled, y_train, classes=np.array([0, 1]))
        predictions = model.predict(X_valid_scaled)
        scores.append(accuracy_score(y_valid, predictions))
    accuracy_history[schedule_name] = scores

# Print concise context for the plotted comparison.
print("scikit-learn version:", sklearn.__version__)
print("Final constant accuracy:", round(accuracy_history["constant"][-1], 3))
print("Final adaptive accuracy:", round(accuracy_history["adaptive"][-1], 3))

# Plot how each schedule behaves over repeated epochs.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, accuracy_history["constant"], marker="o", label="constant")
ax.plot(epochs, accuracy_history["adaptive"], marker="s", label="adaptive")
ax.set_title("SGD learning rate schedules")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0.5, 1.0)
ax.legend()
plt.show()



### **2.3. Early Stopping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_02_03.jpg?v=1783996975" width="250">



>* Stops SGD models from overfitting training data
>* Uses validation performance to decide when to stop

>* Track validation performance on held-out data
>* Use patience to avoid premature stopping

>* Learning rate affects early stopping reliability
>* Saves computation while improving generalization



In [ ]:
#@title Python Code - Early Stopping

# Demonstrate early stopping for SGD training.
# Track validation accuracy after each epoch.
# Stop when improvement has clearly stalled.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small deterministic binary classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

# Split data before scaling to avoid leakage.
train_features, valid_features, train_target, valid_target = train_test_split(
    features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Fit scaling only on the training data.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
valid_scaled = scaler.transform(valid_features)

# Validate the basic shapes before training.
if train_scaled.shape[0] != train_target.shape[0]:
    raise ValueError("Training features and labels must align.")

# Configure patience and minimum meaningful improvement.
max_epochs = 40
patience = 4
min_improvement = 0.001

# Use one SGD classifier updated one epoch at a time.
model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=0.0005,
    learning_rate="optimal",
    random_state=42,
)

best_accuracy = 0.0
epochs_without_improvement = 0
validation_history = []
classes = np.array([0, 1])

# Stop after patience epochs without validation improvement.
for epoch in range(1, max_epochs + 1):
    model.partial_fit(train_scaled, train_target, classes=classes)
    predictions = model.predict(valid_scaled)
    validation_accuracy = accuracy_score(valid_target, predictions)
    validation_history.append(validation_accuracy)

    if validation_accuracy > best_accuracy + min_improvement:
        best_accuracy = validation_accuracy
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement = epochs_without_improvement + 1

    if epochs_without_improvement >= patience:
        stopped_epoch = epoch
        break
else:
    stopped_epoch = max_epochs

print("scikit-learn version:", sklearn.__version__)
print("Best validation accuracy:", round(best_accuracy, 3))
print("Best epoch:", best_epoch)
print("Stopped after epoch:", stopped_epoch)

# Plot validation accuracy to show the stopping decision.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(validation_history) + 1), validation_history, marker="o")
ax.axvline(best_epoch, color="green", linestyle="--", label="best epoch")
ax.axvline(stopped_epoch, color="red", linestyle=":", label="stopped")

ax.set_title("Early stopping monitors validation accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.legend()
plt.show()



## **3. Incremental Learning**

### **3.1. Incremental Model Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_03_01.jpg?v=1783996969" width="250">



>* Learn from small batches over time
>* Update models as new data arrives

>* New batches update existing model knowledge
>* Balanced data and known classes matter

>* Monitor updates to prevent harmful model drift
>* Use checkpoints and mini-batches for stability



In [ ]:
#@title Python Code - Incremental Model Updates

# This example updates one classifier incrementally.
# Mini-batches simulate data arriving over time.
# Accuracy shows learning after each update.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# A small generated dataset keeps the lesson self-contained.
features, labels = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

# The split keeps final evaluation separate from training.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Scaling is fitted only on the training data.
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

# This validation catches unexpected dataset shape problems.
if train_features.shape[0] != train_labels.shape[0]:
    raise ValueError("Training features and labels must align.")

# SGDClassifier supports partial_fit for incremental updates.
model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    learning_rate="optimal",
    max_iter=1,
    tol=None,
    random_state=42,
)

classes = np.array([0, 1])
batch_size = 75
batch_starts = range(0, train_features.shape[0], batch_size)

print("scikit-learn version:", sklearn.__version__)
print("Update | Seen samples | Test accuracy")

# Each partial_fit call updates the existing model.
for update_number, start in enumerate(batch_starts, start=1):
    stop = start + batch_size
    batch_features = train_features[start:stop]
    batch_labels = train_labels[start:stop]

    if update_number == 1:
        model.partial_fit(batch_features, batch_labels, classes=classes)
    else:
        model.partial_fit(batch_features, batch_labels)

    predictions = model.predict(test_features)
    accuracy = accuracy_score(test_labels, predictions)
    seen_samples = min(stop, train_features.shape[0])

    print(update_number, "|", seen_samples, "|", round(accuracy, 3))

print("Final model was updated without retraining from scratch.")



### **3.2. Sparse Input Training**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_03_02.jpg?v=1783996973" width="250">



>* Sparse data stores only present features
>* SGD efficiently trains high-dimensional linear models

>* Update models with compact sparse batches
>* Keep streaming workflows scalable and consistent

>* Preserve sparsity to avoid memory overload
>* Regularize and monitor incremental sparse training



In [ ]:
#@title Python Code - Sparse Input Training

# Sparse batches keep high-dimensional training memory efficient.
# Partial fit updates one compact batch at a time.
# Accuracy confirms the incremental sparse classifier learned.

import numpy as np
import sklearn
from scipy import sparse
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Create deterministic sparse text-like feature counts.
rng = np.random.default_rng(42)
samples = 600
features = 2000

# Each example uses only a few nonzero features.
rows = np.repeat(np.arange(samples), 12)
cols = rng.integers(0, features, size=samples * 12)
values = rng.integers(1, 4, size=samples * 12)

# Build a compressed sparse row matrix for efficient learning.
X_sparse = sparse.csr_matrix((values, (rows, cols)), shape=(samples, features))
positive_signal = X_sparse[:, :40].sum(axis=1).A.ravel()
negative_signal = X_sparse[:, 40:80].sum(axis=1).A.ravel()

y = (positive_signal >= negative_signal).astype(int)
if len(np.unique(y)) != 2:
    raise ValueError("The generated labels need two classes.")

# Split before training to keep evaluation honest.
X_train, X_test, y_train, y_test = train_test_split(
    X_sparse, y, test_size=0.25, stratify=y, random_state=42
)

# SGDClassifier supports sparse input and incremental partial_fit.
model = SGDClassifier(
    loss="log_loss", penalty="l2", alpha=0.0001, random_state=42, max_iter=1
)
classes = np.array([0, 1])

# Feed the model small sparse batches instead of one dense matrix.
batch_size = 75
for start in range(0, X_train.shape[0], batch_size):
    stop = start + batch_size
    model.partial_fit(X_train[start:stop], y_train[start:stop], classes=classes)

# Compare sparse storage with the dense size it avoids.
sparse_mb = (
    X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes
) / 1_000_000

dense_mb = X_train.toarray().nbytes / 1_000_000
accuracy = accuracy_score(y_test, model.predict(X_test))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Training shape: {X_train.shape[0]} rows x {X_train.shape[1]} features")
print(f"Nonzero training entries: {X_train.nnz}")
print(f"Sparse storage: {sparse_mb:.2f} MB")
print(f"Equivalent dense storage: {dense_mb:.2f} MB")
print(f"Test accuracy after partial_fit batches: {accuracy:.3f}")



### **3.3. Online Anomaly Detection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_03_03.jpg?v=1783996971" width="250">



>* Detect anomalies immediately in streaming data
>* Update models incrementally without full retraining

>* Detect deviations from learned normal behavior
>* Sparse updates handle large changing data efficiently

>* Balance model stability with adaptability
>* Use feedback to guide incremental updates



In [ ]:
#@title Python Code - Online Anomaly Detection

# This example streams sparse event data.
# SGD learns normal behavior incrementally.
# Large prediction errors flag anomalies.

import numpy as np
import sklearn
from scipy import sparse
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error

# A fixed generator makes the stream reproducible.
rng = np.random.default_rng(42)
n_samples = 240
n_features = 1000

# Each row has only a few active event indicators.
rows = np.repeat(np.arange(n_samples), 8)
cols = rng.integers(0, n_features, size=n_samples * 8)
values = rng.normal(1.0, 0.2, size=n_samples * 8)

# Sparse CSR stores only nonzero values efficiently.
X_stream = sparse.csr_matrix(
    (values, (rows, cols)), shape=(n_samples, n_features)
)
true_weights = rng.normal(0.0, 0.4, size=n_features)

# Most targets follow the normal pattern.
y_stream = X_stream @ true_weights + rng.normal(0.0, 0.2, size=n_samples)
anomaly_indices = np.array([70, 135, 210])
y_stream[anomaly_indices] += np.array([6.0, -7.0, 5.5])

# Validate the sparse stream before modeling.
if X_stream.shape != (n_samples, n_features):
    raise ValueError("The sparse stream has an unexpected shape.")

# The first batch teaches the model normal behavior.
initial_size = 40
model = SGDRegressor(
    loss="squared_error", penalty="l2", alpha=0.0001,
    learning_rate="constant", eta0=0.01, random_state=42
)

# partial_fit updates the model without retraining from scratch.
model.partial_fit(X_stream[:initial_size], y_stream[:initial_size])
errors = []
flags = []

# Each new observation is scored, then used for learning.
for i in range(initial_size, n_samples):
    prediction = model.predict(X_stream[i])[0]
    error = abs(y_stream[i] - prediction)
    errors.append(error)
    flags.append(error > 3.0)
    model.partial_fit(X_stream[i], [y_stream[i]])

# Summarize how online scoring found unusual observations.
flagged_positions = np.where(np.array(flags))[0] + initial_size
mae = mean_absolute_error(y_stream[initial_size:], model.predict(X_stream[initial_size:]))

print("scikit-learn version:", sklearn.__version__)
print("Sparse matrix shape:", X_stream.shape)
print("Stored nonzero values:", X_stream.nnz)
print("Known anomaly rows:", anomaly_indices.tolist())
print("Flagged anomaly rows:", flagged_positions.tolist())
print("Final stream MAE:", round(mae, 3))



# <font color="#418FDE" size="6.5" uppercase>**Stochastic Gradient**</font>


In this lecture, you learned to:
- Train SGD-based classifiers and regressors with appropriate losses and penalties. 
- Configure learning rates, early stopping, and regularization for stable training. 
- Use partial_fit and sparse inputs for incremental or large-scale workflows. 

In the next Module (Module 12), we will go over 'SVMs Kernel Ridge'